In [1]:
from functools import partial
import json
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
from swanlab.integration.transformers import SwanLabCallback
import swanlab
from modelscope import AutoTokenizer

from peft import LoraConfig, get_peft_model, TaskType

import torch
import random
import numpy as np
import os

seed = 7
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

g:\software\anaconda\envs\pytorch\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
model_id = "qwen2.5-1.5B-Instruct"  
model_path = "G:/model_weights/models/model/qwen2.5-1.5B-Instruct"
data_path = "D:/doc/my_project/lianxi/home_work/stage_3/data_set"

# 1.数据预处理

In [3]:
class TCMNERDataset():
    def __init__(self, data_path):
        self.label_set = set()
        self.data = self.load_data(data_path)
        # self.dataset = Dataset.from_list(self.data)
        self.id2label = {idx: item for idx, item in enumerate(self.label_set)}
        self.label2id = {item: idx for idx, item in self.id2label.items()}


    def load_data(self, data_path):
        dataset = []
        with open(data_path, 'r', encoding='utf-8') as f:
            trunk = f.read().split('\n\n')
            for line in trunk:
                input, output = '', []
                for idx, string in enumerate(line.split('\n')):
                    if not string.strip():  # 跳过空行
                        continue
                    text, tag = string.split(' ')  # 假设文本和标签之间用空格分隔
                    input += text
                    if tag.startswith('B-'):
                        self.label_set.add(tag[2:])
                        output.append([text, tag[2:]])  # 添加标签内容
                    elif tag.startswith('I-') and output:
                        output[-1][2] += text  # 更新标签内容
                dataset.append({'input': input, 'output': json.dumps(output, ensure_ascii=False)})  # 将标签列表转换为JSON字符串
        return dataset

In [4]:
from pathlib import Path
train_data_path = Path(data_path) / 'medical.train'
test_data_path = Path(data_path) / 'medical.test'
dev_data_path = Path(data_path) / 'medical.dev'

In [5]:
train_data = TCMNERDataset(train_data_path)
test_data = TCMNERDataset(test_data_path)
dev_data = TCMNERDataset(dev_data_path)

IndexError: list index out of range

In [34]:

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)#, id2label=train_data.id2label, label2id=train_data.label2id)
model.enable_input_require_grads()  # 开启梯度检查点时，要执行该方法

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [35]:
train_data.data[0], train_data.id2label, train_data.label2id

({'input': '现头昏口苦', 'output': '[[3, 4, "口苦", "临床表现"]]'},
 {0: 'B-西医治疗',
  1: 'I-方剂',
  2: 'I-中医诊断',
  3: 'I-中医治疗',
  4: 'B-其他治疗',
  5: 'B-中医治疗',
  6: 'I-西医治疗',
  7: 'I-中药',
  8: 'B-中医治则',
  9: 'I-临床表现',
  10: 'B-方剂',
  11: 'I-西医诊断',
  12: 'B-中医证候',
  13: 'B-西医诊断',
  14: 'B-中医诊断',
  15: 'I-中医治则',
  16: 'B-临床表现',
  17: 'B-中药',
  18: 'I-其他治疗',
  19: 'I-中医证候'},
 {'B-西医治疗': 0,
  'I-方剂': 1,
  'I-中医诊断': 2,
  'I-中医治疗': 3,
  'B-其他治疗': 4,
  'B-中医治疗': 5,
  'I-西医治疗': 6,
  'I-中药': 7,
  'B-中医治则': 8,
  'I-临床表现': 9,
  'B-方剂': 10,
  'I-西医诊断': 11,
  'B-中医证候': 12,
  'B-西医诊断': 13,
  'B-中医诊断': 14,
  'I-中医治则': 15,
  'B-临床表现': 16,
  'B-中药': 17,
  'I-其他治疗': 18,
  'I-中医证候': 19})

In [36]:
train_dataset = Dataset.from_list(train_data.data)
test_dataset = Dataset.from_list(test_data.data)
dev_dataset = Dataset.from_list(dev_data.data)

# 2.构造训练数据

In [37]:
def preprocess_function(tokenizer, examples):
    MAX_LENGTH = 512
    system_prompt = "你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[[', 4, '口苦', '临床表现'],[5, 6, '口干', '临床表现']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": examples['input']},
    ]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,) # 输出会停在最后一个user，然后添加assistant的开始标记等待模型生成, 不会编码assistant的内容
    # print(messages)
    # print(examples)
    response = tokenizer(examples['output'], add_special_tokens=False)
    inputs = tokenizer(messages, truncation=True, add_special_tokens=False)

    input  = inputs['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]  # 将输入和输出拼接在一起，并添加结束标记

    attention_mask = inputs['attention_mask'] + response['attention_mask'] + [1]  # 拼接注意力掩码

    outputs = [-100] * len(inputs['input_ids']) + response['input_ids'] + [tokenizer.eos_token_id]  # 输出标签的token id，并添加结束标记

    if len(input) > MAX_LENGTH:
        input = input[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        outputs = outputs[:MAX_LENGTH]

    return {'input_ids': input, 'attention_mask': attention_mask, 'labels': outputs}

p_preprocess = partial(preprocess_function, tokenizer)

messages:
```<|im_start|>system
你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[[', 4, '口苦', '临床表现'],[5, 6, '口干', '临床表现']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。<|im_end|>
<|im_start|>user
现头昏口苦<|im_end|>
<|im_start|>assistant
```

In [ ]:
train_dataset[0]
preprocess_function(tokenizer, train_dataset[0])

In [31]:
tokenizer.decode(preprocess_function(tokenizer, train_dataset[0])['input_ids'])

'<|im_start|>system\n你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[[\', 4, \'口苦\', \'临床表现\'],[5, 6, \'口干\', \'临床表现\']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。<|im_end|>\n<|im_start|>user\n现头昏口苦<|im_end|>\n<|im_start|>assistant\n[[3, 4, "口苦", "临床表现"]]<|endoftext|>'

In [17]:
train_datasets = train_dataset.map(p_preprocess, remove_columns=train_dataset.column_names, num_proc=4)
# test_datasets = test_dataset.map(p_preprocess, remove_columns=test_dataset.column_names, num_proc=4)
dev_datasets = dev_dataset.map(p_preprocess, remove_columns=dev_dataset.column_names, num_proc=4)

Map (num_proc=4):   0%|          | 0/5259 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/657 [00:00<?, ? examples/s]

#  3.训练

In [12]:
fine_tuning_config = LoraConfig(task_type=TaskType.CAUSAL_LM, 
                                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                r=8, 
                                lora_alpha=16, 
                                inference_mode=False, 
                                lora_dropout=0.1)

peft_model = get_peft_model(model, fine_tuning_config)
peft_model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [13]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
  

In [18]:
args = TrainingArguments(
    output_dir="G:/model_weights/models/checkpoint/qwen2.5-1.5-tcm-ner",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, # 积累多少batch后更新一次权重，主要用于解决显存不足的问题
    eval_strategy="epoch",
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [15]:
swanlab_callback = SwanLabCallback(
    project="Qwen2.5-NER-fintune",
    experiment_name="Qwen2-1.5B-Instruct",
    description="使用通义千问Qwen2-1.5B-Instruct模型在中医药NER数据集上微调，实现关键实体识别任务。",
    config={
        "model": model_id,
        "model_dir": model_path,
        "dataset": "qgyd2021/chinese_ner_sft",
    },
)

In [19]:
trainer = Trainer(
    model=peft_model,
    args=args,
    train_dataset=train_datasets,
    eval_dataset=dev_datasets,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)


In [20]:
trainer.train()

Output()

Output()

swanlab: Tracking run with swanlab version 0.7.8

swanlab: Run data will be saved locally in 
d:\doc\my_project\lianxi\home_work\stage_3\swanlog\run-20260221_200835-ily2vcbjlf3ixxfg333l2

swanlab: 👋 Hi yuerg1230,welcome to swanlab!

swanlab: Syncing run Qwen2-1.5B-Instruct to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune

swanlab: 🚀 View run at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune/runs/ily2vcbjlf3ixxfg333l2

Epoch,Training Loss,Validation Loss
1,0.204107,0.219211
2,0.171429,0.187878


TrainOutput(global_step=658, training_loss=0.2346966407944004, metrics={'train_runtime': 965.9902, 'train_samples_per_second': 10.888, 'train_steps_per_second': 0.681, 'total_flos': 1.7883584231021568e+16, 'train_loss': 0.2346966407944004, 'epoch': 2.0})

样本数量是5259，训练的时候进度提示总共是658，原因？\
`per_device_train_batch_size` = 4 每次训练4个样本 \
`gradient_accumulation_steps` = 4 每4个batch更新一次权重 \
`num_train_epochs`=2              总共训练两轮 \
训练器显示的步数是权重更新的次数\
$5259 \div 4 \div 4 \times 2 \approx 658$

In [21]:
swanlab.finish()

swanlab: 🏠 View project at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune

swanlab: 🚀 View run at https://swanlab.cn/@yuerg1230/Qwen2.5-NER-fintune/runs/ily2vcbjlf3ixxfg333l2

# 4.加载检查点Lora

In [6]:
from peft import  PeftModel
base_model = AutoModelForCausalLM.from_pretrained(model_path).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_path)
peft_model = PeftModel.from_pretrained(base_model, "G:/model_weights/models/checkpoint/qwen2.5-1.5-tcm-ner/checkpoint-658").cuda()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [7]:
from pathlib import Path
test_data_path = Path(data_path) / 'medical.test'

test_data = TCMNERDataset(test_data_path)

In [8]:
def f1_score(set_a, set_b):
    """ 严格的F1 分数计算，只有当索引位置、实体、标签预测完全正确时才算TP，否则算FP和FN """
    TP = len(set_a & set_b)
    FP = len(set_b - set_a)
    FN = len(set_a - set_b)
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return f1


## 批量样本推断

In [ ]:
set_a_strict = set()
set_b_strict = set()
set_a_relax = set()
set_b_relax = set()


tag_dict = {}
for entity in test_data.label_set:
    tag_dict[entity] = [set(), set()]  # [label, label_predicted]

def predict_collect(predictions, references):
    """ 预测结果收集函数，返回严格匹配和宽松匹配的实体集合 """
    set_a_strict = set()  # 严格匹配：索引位置、实体内容、标签类型都必须完全匹配
    set_b_strict = set()  # 模型预测的严格匹配结果
    
    for item in references:
        item = json.loads(item)
        for label in item:
            tag_dict[label[2]][0].add(tuple(label))  # 仅比较实体内容和标签类型，忽略位置
            set_a_strict.add(tuple(label))

    for output in predictions:
        try:
            output = output.strip().replace('\n', '').replace('“', '"').replace('”', '"').replace('{', '').replace('}', '')
            output = json.loads(output)  # 去除多余的换行符和空格

            for item in output:
                if not item or len(item) != 2:  # 如果输出是空列表，跳过
                    continue
                if item[1] in tag_dict:
                    tag_dict[item[1]][1].add(tuple(item))
                set_b_strict.add(tuple(item))
        except:
            import traceback
            print(references)
            print('--1')
            print(output)
            print('--2')
            traceback.print_exc()
    
    return set_a_strict, set_b_strict,

In [ ]:
import tqdm

from torch.utils.data import DataLoader

test_dataset = DataLoader(test_data.data, batch_size=32, shuffle=False)

system_prompt = "你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[['口苦', '临床表现'],['口干', '临床表现']]，其中每个元素分别表示命名体的文本内容和标签类型。如果没有则返回空列表[]"

pbar = tqdm.tqdm(total=len(test_dataset), desc="Evaluating on test set")

for batch_sample in test_dataset:
    inputs = []
    for i in batch_sample['input']:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": i},
        ]
        input = tokenizer.apply_chat_template(messages, padding_side='left', tokenize=False, add_generation_prompt=True) # 输出会停在最后一个user，然后添加assistant的开始标记等待模型生成, 不会编码assistant的内容
        inputs.append(input)
    inputs = tokenizer(inputs, padding_side='left', padding=True, truncation=True, return_tensors="pt").to(peft_model.device)

    with torch.no_grad():
        outputs = peft_model.generate(inputs.input_ids, max_new_tokens=512)
    
    output = outputs[:, inputs.input_ids.shape[1]:]  # 获取生成的部分
    decoded_outputs = tokenizer.batch_decode(output, skip_special_tokens=True)
    
    set_a_strict, set_b_strict = predict_collect(decoded_outputs, batch_sample['output'])

In [74]:
len(set_a_strict), count, len(set_b_strict)

(1345, 1380, 1274)

In [75]:
f1_score(set_a_strict, set_b_strict), f1_score(set_a_relax, set_b_relax)

(0.05574646811760214, 0.40548970679975044)

## 单样本推断

In [ ]:
import tqdm
system_prompt = "你是一个中医药领域的命名体识别专家，你需要从给定句子中提取相关命名体，以json格式输出，例如：[[2, 4, '口苦', '临床表现'],[5, 6, '口干', '临床表现']]，其中每个元素分别表示命名体的起始位置、结束位置、文本内容和标签类型。如果没有则返回空列表[]"

set_a_strict = set()
set_b_strict = set()
set_a_relax = set()
set_b_relax = set()

pbar = tqdm.tqdm(total=len(test_data.data), desc="Evaluating on test set")

for i in test_data.data:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{i['input']}"},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # 输出会停在最后一个user，然后添加assistant的开始标记等待模型生成, 不会编码assistant的内容
    inputs = tokenizer([inputs], return_tensors="pt").to(peft_model.device)
    with torch.no_grad():
        outputs = peft_model.generate(inputs.input_ids, max_new_tokens=512)
    
    output = outputs[0][len(inputs.input_ids[0]):]  # 获取生成的部分
    decoded_output = tokenizer.decode(output, skip_special_tokens=True)
    decoded_output = json.loads(decoded_output.strip().replace('\n', ''))  # 去除多余的换行符和空格
    print(decoded_output)
    print(i['output'])


    for item in json.loads(i['output']):
        set_a_strict.add(tuple(item))
        set_a_relax.add((item[2], item[3]))  # 仅比较实体内容和标签类型，忽略位置

    try:
        for item in decoded_output:
            set_b_strict.add(tuple(item))
            set_b_relax.add((item[2], item[3]))  # 仅比较实体内容和标签类型，忽略位置
    except:
        import traceback
        print(i['output'])
        print(decoded_output)
        traceback.print_exc()
    finally:
        pbar.update(1)


In [9]:
del peft_model

NameError: name 'peft_model' is not defined

In [10]:
import gc
gc.collect()

9352

In [11]:
torch.cuda.empty_cache()